In [1]:
# Code for Verdigris river study case for 2019 event
# Updated version for libraries generated with FLDPLN v8
# Updated folder paths and data!
import sys
import time
from dask.distributed import Client, LocalCluster
from dask import visualize

In [2]:
# Tool/script folder
fldplnToolFolder = r'C:/Users/hobbe/Documents/SummerInsitute/FLDSensing/Verdigris_v8/source' # tool development folder, has the latest version
# Add the tool/script folder to sys.path to access fldpln modules
sys.path.append(fldplnToolFolder) 

# fldpln module
from fldpln import *
from fldpln_library import *
from fldpln_gauge import *

In [3]:
# tiled library folder
libFolder =  r'C:\Users\hobbe\Documents\SummerInsitute\FLDSensing\verdigris_10m_v8\tiled_snz_library'

# libraries to be mapped
allLibNames = ['lib_fldsensing']

# Set output folder, not used yet in this code version
outputFolder = r'C:\Users\hobbe\Documents\SummerInsitute\FLDSensing\verdigris_10m_v8\tiled_snz_library'  

In [7]:
# All Gauges in verdigris extent
gaugeStageFileName = 'VerdigrisGauges.xlsx' 
sheetName = '2019Flood' # all gauges
# sheetName = 'TwoDsGauges' # 2 downstream gauges
# sheetName = 'MSTK1' # the last downstream gauge used in HEC-RAS model

# read gauge file
gaugeStages = pd.read_excel(gaugeStageFileName, sheet_name=sheetName) 
# Need to calculate gauge stage elevation if necessary!

# keep only necessary fields from gauges
keptFields = ['stationid','x','y','stage_elevation']
gaugeWithStageElevations = gaugeStages[keptFields]
print(gaugeWithStageElevations)

        stationid              x             y  stage_elevation
0        07170990  803191.638090  4.100896e+06       216.833560
1           CFVK1  799235.783244  4.106982e+06       221.251064
2  07170500,IDPK1  794776.617849  4.124860e+06       232.392504
3  07169500,FRNK1  779884.869637  4.155983e+06       259.482128
4  07166500,ATOK1  793849.397926  4.158843e+06       249.503184


### Synthetic Gauge Stage from the National Water Model and HAND

HAND FIM uses NWM's discharge and turn it into stage. Here we use HAND reach stage to run FLDPLN for the event. Concepually, we turn reach stage into a synthetic gauge located at the either the mid-point or the outlet of the reach. Selecting the HAND reaches and sythteric gauge location is done by graduate student David Weiss manually for the Wildcat Creek example. Those sytheteic gauges can be treated as USGS/AHPS guages. The key fields needed are: stationid, x, y, and stage_elevation.  Note that we assume the HAND reach stage elevation is the same as the FLDPLN library DEM. 

In [8]:
# All Gauges in verdigris extent
gaugeStageFileName = 'VerdigrisGauges.xlsx' 
sheetName = 'Low90Percentile' # all gauges
# sheetName = 'TwoDsGauges' # 2 downstream gauges
# sheetName = 'MSTK1' # the last downstream gauge used in HEC-RAS model

# read gauge file
gaugeStages = pd.read_excel(gaugeStageFileName, sheet_name=sheetName) 
# Need to calculate gauge stage elevation if necessary!

# keep only necessary fields from gauges
keptFields = ['stationid','x','y','stage_elevation']
gaugeWithStageElevations = gaugeStages[keptFields]
print(gaugeWithStageElevations)

     stationid         x          y  stage_elevation
0            1  803087.5  4106422.5       222.448783
1            2  802277.5  4106582.5       222.693773
2            3  792982.5  4137752.5       241.172213
3            4  794377.5  4140217.5       242.002957
4            5  794977.5  4124402.5       232.614884
..         ...       ...        ...              ...
318        319  801507.5  4107747.5       216.729867
319        320  802402.5  4113242.5       223.083972
320        321  798132.5  4102832.5       217.824431
321        322  811457.5  4111812.5       225.341318
322        323  795537.5  4144347.5       233.526116

[323 rows x 4 columns]


### Snap Gauges to FSPs and Calculate Gauge DOF

Here we snap gauges (with their stage elevation) to FLDPLN flood source pixels (FSPs), which are the stream pixels. Each snapped gauge FSP has a stream bed elevaltion, which is used to claculate the depth of flow/flood (DOF) at those FSPs. 

This process also identifies the FLDPLN libraries that the gauges belong to. Note that the same gauges can be snapped to more than one library as FLDPLN libraries may overlap and the overalpping FSPs may have different coordinates! 

In [9]:
# snap gauges to FSPs on-the-fly
print('Snap gauges to FSPs ...')
print(f'Number of gauges: {len(gaugeWithStageElevations.index)}')

# FLDPLN libraries to whose FSPs gauges are sanpped. All the libraries by default but can be a subset
libs2Map = ['lib_fldsensing']

# snap the gauges to FSPs. 
# Fields 'StrOrd','DsDist','SegId','FilledElev'are used for interpolating other FSP DOF
# Note that 'lib_name','FspX', 'FspY' together uniquely identify a FSP (as there are overlapping FSPs between libraries)!
gaugeFspDf = SnapGauges2Fsps(libFolder,libs2Map,gaugeWithStageElevations,snapDist=350,gaugeXField='x',gaugeYField='y',fspColumns=['FspX','FspY','StrOrd','DsDist','SegId','FilledElev']) 
print(gaugeFspDf)

# calculate gauge FSP's DOF
gaugeFspDf['Dof'] = gaugeFspDf['stage_elevation'] - gaugeFspDf['FilledElev']

# keep only necessary columns for gauge FSPs
gaugeFspDf = gaugeFspDf[['lib_name','FspX','FspY','StrOrd','DsDist','SegId','FilledElev','Dof']] # Note that 'lib_name','FspX', 'FspY' together uniquely identify a FSP!!!

# show info
print(f'Number of snapped gauge FSPs: {len(gaugeFspDf)}')
# Find libs where the gauges are snapped to, and they are the actual libs to map
libs2Map = gaugeFspDf['lib_name'].drop_duplicates().tolist()
print(f'Libraries gauges snapped to: {libs2Map}')
print(gaugeFspDf)

#
# save snapped gauges to CSV file for checking
# gaugeFspDf.to_csv(os.path.join(outputFolder, 'SnappedGauges.csv'), index=False)

Snap gauges to FSPs ...
Number of gauges: 323


C:\Users/hobbe/Documents/SummerInsitute/FLDSensing/Verdigris_v8/source\fldpln.py:93: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  nearestP2Df = pd.concat([nearestP2Df,t],ignore_index=False)


     index  stationid         x          y  stage_elevation  d2NearestFsp  \
0        0          1  803087.5  4106422.5       222.448783      3.535534   
1        1          2  802277.5  4106582.5       222.693773      7.905694   
2        2          3  792982.5  4137752.5       241.172213     12.747549   
3        3          4  794377.5  4140217.5       242.002957     12.747549   
4        4          5  794977.5  4124402.5       232.614884      7.905694   
..     ...        ...       ...        ...              ...           ...   
318    318        319  801507.5  4107747.5       216.729867      3.535534   
319    319        320  802402.5  4113242.5       223.083972      7.905694   
320    320        321  798132.5  4102832.5       217.824431      3.535534   
321    321        322  811457.5  4111812.5       225.341318     12.747549   
322    322        323  795537.5  4144347.5       233.526116      3.535534   

         FspX       FspY  StrOrd        DsDist  SegId  FilledElev  \
0    8

## Interpolate FSP's DOF

Here we interpolate the DOF for all the FSPs between the gauge-FSPs using their DOF calculated from previous step. The interpolation uses stream orders and starts from low stream order (i.e., main streams) to high stream order (i.e., tributatried). Either horizontal or vertical (by defaut) interpolation can be used.

In [10]:
# Find libs with snapped gauges. They are the actual libs to map
libs2Map = gaugeFspDf['lib_name'].drop_duplicates().tolist()

# prepare the DF for storing interpolated FSP DOF
fspDof = pd.DataFrame(columns=['LibName','FspId','Dof'])

# prepare DFs for saving interpolated FSPs and their segment IDs
fspCols = fspInfoColumnNames + ['Dof']
segIdCols = ['SegId','LibName']
fsps = pd.DataFrame(columns=fspCols)
segIds =pd.DataFrame(columns=segIdCols)

# map each library
for libName in libs2Map:
    # interpolate DOF for the gauges
    # print('Interpolate FSP DOF using gauge DOF ...')
    # fspIdDof = InterpolateFspDofFromGauge(libFolder,libName,gaugeFspDf) # 'V' by default
    fspIdDof = InterpolateFspDofFromGauge(libFolder,libName,gaugeFspDf,weightingType='H') # horizontal interpolation
    fspIdDof['LibName'] = libName
    # fspDof = fspDof.append(fspIdDof[['LibName','FspId','Dof']], ignore_index=True)
    fspDof = pd.concat([fspDof,fspIdDof[['LibName','FspId','Dof']]], ignore_index=True)

    # Keep interpolated FSP DOF for saving later
    fspFile = os.path.join(libFolder, libName, fspInfoFileName)
    fspDf = pd.read_csv(fspFile) 
    fspDf = pd.merge(fspDf,fspDof,how='inner',on=['FspId'])
    # fsps = fsps.append(fspDf, ignore_index=True)
    fsps = pd.concat([fsps,fspDf], ignore_index=True)
    
    # Keep FSP segment IDs for saving later
    t =  pd.DataFrame(fspDf['SegId'].drop_duplicates().sort_values())
    t['LibName'] = libName
    # segIds = segIds.append(t, ignore_index=True)
    segIds = pd.concat([segIds,t], ignore_index=True)

# show interpolated FSPs with Dof
print(fspDof)

#
# save interpolated FSP DOF and their segments for checking. This block of code should be commented out if no-checking needed
#
# Save DOF and segment IDs to CSV files
FspDofFile = os.path.join(outputFolder, 'Interpolated_FSP_DOF.csv')
SegIdFile = os.path.join(outputFolder, 'Interpolated_SegIds.csv')
fsps.to_csv(FspDofFile, index=False)
segIds.to_csv(SegIdFile, index=False)

# # turn interpolated sgements into a shapefile
# for libName in libs2Map:
#     segShp = os.path.join(libFolder, libName, 'stream_orders.shp')
#     segs = gpd.read_file(segShp)
#     segs['LibName'] = libName
#     # print(segs)
#     # join by two fields: SegId and LibName
#     segDf = pd.merge(segs,segIds,how='inner',on=['SegId','LibName'])
#     # print(segDf)
#     # write segments as a shapefile
#     segDf.to_file(os.path.join(outputFolder, 'Interpolated_Segements.shp'))

              LibName FspId       Dof
0      lib_fldsensing  9692  7.094885
1      lib_fldsensing  9693  7.111797
2      lib_fldsensing  9694  7.128709
3      lib_fldsensing  9695  7.152627
4      lib_fldsensing  9696  7.169539
...               ...   ...       ...
33770  lib_fldsensing   712  6.984916
33771  lib_fldsensing   713  6.982664
33772  lib_fldsensing   714  6.980413
33773  lib_fldsensing   715  6.978162
33774  lib_fldsensing   716  6.975911

[33775 rows x 3 columns]


C:\Users/hobbe/Documents/SummerInsitute/FLDSensing/Verdigris_v8/source\fldpln.py:523: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fspDof = pd.concat([fspDof,fspOrd],ignore_index=True)
C:\Users\hobbe\AppData\Local\Temp\ipykernel_9048\3740564350.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fspDof = pd.concat([fspDof,fspIdDof[['LibName','FspId','Dof']]], ignore_index=True)
C:\Users\hobbe\AppData\Local\Temp\ipykernel_9048\3740564350.py:28: FutureWarning: The behavior of DataFrame concatena

## Map Flood Inundation Depth


### Set Mapping Parameters

Setup the map folder (i.e., outMapFolderName) which is under the output folder and comtains all inundation depth maps. Additional settings include whether to mosaic tiles as single COG file and whether use a Dask local cluster to speed up the mapping.

In [11]:
# set up map folder
outMapFolderName = 'Verdigris_May272019_NWM_Low90Percentile'

# Create folders for storing temp and output map files
outMapFolder,scratchFolder = CreateFolders(outputFolder,'scratch',outMapFolderName)

# whether mosaci tiles as a single COG
mosaicTiles = True #True #False

# Using LocalCluster by default
useLocalCluster = False # This doesn't work on my office desktop though it works fine on KBS server
numOfWorkers = round(0.8*os.cpu_count())
numOfWorkers = 6
print(f'Number of workers: {numOfWorkers}')

Number of workers: 6


In [12]:
# show mapping info
print(f'Tiled FLDPLN library folder: {libFolder}')
print(f'Map folder: {outMapFolder}')
# Find libs needs mapping
libs2Map = fspDof['LibName'].drop_duplicates().tolist()
print(f'Libraries to map: {libs2Map}')

# check running time
startTimeAllLibs = time.time()

# create a local cluster to speed up the mapping. Must be run inside "if __name__ == '__main__'"!!!
if useLocalCluster:
    # cluster = LocalCluster(n_workers=4,processes=False)
    try:
        print('Start a LocalCluster ...')
        # NOTE: set worker space (i.e., local_dir) to a folder that the LocalCluster can access. When run the script through a scheduled task, 
        # the system uses C:\Windows\system32 by default, which a typical user doesn't have the access!
        # cluster = LocalCluster(n_workers=numOfWorkers,memory_limit='32GB',local_dir="D:/projects_new/fldpln/tools") # for KARS production server (192G RAM & 8 cores)
        # cluster = LocalCluster(n_workers=numOfWorkers,processes=False) # for KARS production server (192G RAM & 8 cores)
        cluster = LocalCluster(n_workers=numOfWorkers,memory_limit='8GB',local_dir="E:\temp") # for office desktop (64G RAM & 8 cores)
        # print('Watch workers at: ',cluster.dashboard_link)
        print(f'Number of workers: {numOfWorkers}')
        client = Client(cluster)
        # print scheduler info
        # print(client.scheduler_info())
    except:
        print('Cannot create a LocalCLuster!')
        useLocalCluster = False

# dict to store lib processing time
libTime={}

# map each library
for libName in libs2Map:
    # check running time
    startTime = time.time()
    
    # select the FSPs within the lib
    fspIdDof = fspDof[fspDof['LibName']==libName][['FspId','Dof']]

    # mapping flood depth
    if useLocalCluster:
        print(f'Map [{libName}] using LocalCLuster ...')
        # generate a DAG
        dag,dagRoot=MapFloodDepthWithTilesAsDag(libFolder,libName,'snappy',outMapFolder,fspIdDof,aoiExtent=None)
        if dag is None:
            tileTifs = None
        else:
            # visualize DAG
            # visualize(dag)
            # Compute DAG
            tileTifs = client.get(dag, dagRoot)
            if not tileTifs: # list is empty
                tileTifs =  None
    else:
        print(f'Map {libName} ...')
        tileTifs = MapFloodDepthWithTiles(libFolder,libName,'snappy',outMapFolder,fspIdDof,aoiExtent=None)
    print(f'Actual mapped tiles: {tileTifs}')

    # Mosaic all the tiles from a library into one tif file
    if mosaicTiles and not(tileTifs is None):
        print('Mosaic tile maps ...')
        mosaicTifName = libName+'_'+outMapFolderName+'.tif'
        # Simplest implementation, may crash with very large raster
        MosaicGtifs(outMapFolder,tileTifs,mosaicTifName,keepTifs=False)
    
    # check time
    endTime = time.time()
    usedTime = round((endTime-startTime)/60,3)
    libTime[libName] = usedTime
    # print(f'{libName} processing time (minutes):', usedTime)

# Show processing time
# Individual lib processing time
print('Individual library mapping time:', libTime)
# total time
endTimeAllLibs = time.time()
print('Total processing time (minutes):', round((endTimeAllLibs-startTimeAllLibs)/60,3))

#
# Shutdown local clusters
#
if useLocalCluster:
    print('Shutdown LocalCluster ...')
    cluster.close()
    client.shutdown()
    client.close()
    useLocalCluster = False

Tiled FLDPLN library folder: C:\Users\hobbe\Documents\SummerInsitute\FLDSensing\verdigris_10m_v8\tiled_snz_library
Map folder: C:\Users\hobbe\Documents\SummerInsitute\FLDSensing\verdigris_10m_v8\tiled_snz_library\Verdigris_May272019_NWM_Low90Percentile
Libraries to map: ['lib_fldsensing']
Map lib_fldsensing ...
Tiles need to be mapped: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Actual mapped tiles: ['C:\\Users\\hobbe\\Documents\\SummerInsitute\\FLDSensing\\verdigris_10m_v8\\tiled_snz_library\\Verdigris_May272019_NWM_Low90Percentile\\lib_fldsensing_tile_1.tif', 'C:\\Users\\hobbe\\Documents\\SummerInsitute\\FLDSensing\\verdigris_10m_v8\\tiled_snz_library\\Verdigris_May272019_NWM_Low90Percentile\\lib_fldsensing_tile_2.tif', 'C:\\Users\\hobbe\\Documents\\SummerInsitute\\FLDSensing\\verdigris_10m_v8\\tiled_snz_library\\Verdigris_May272019_NWM_Low90Percentile\\lib_fldsensing_tile_3.tif', 'C:\\Users\\hobbe\\Documents\\SummerInsitute\\FLDSensing\\verdigris_10m_v8\\tiled_snz_library\\Verdigris_May2720